# Auditoría de la base imputada

Objetivo: comprobar si `datos_imputados.csv` conserva los datos reales y los extremos, que es lo que necesita la GPD.

Archivos necesarios en la carpeta `data/`:
- `datos_no_imputados.csv`
- `datos_imputados.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("..") / "data"
VARS = ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "PRS",
        "RAINF", "RH", "SO2", "SR", "TOUT", "WSR", "WDR"]

## 1. Cargar las dos bases
La base sin imputar tarda un poco porque pesa ~100 MB.

In [ ]:
ni = pd.read_csv(DATA / "datos_no_imputados.csv", low_memory=False)
im = pd.read_csv(DATA / "datos_imputados.csv")
print("Sin imputar:", ni.shape)
print("Imputada:   ", im.shape)

## 2. ¿En qué columnas están los datos de cada año?
El Excel de 2024 trae los nombres con unidades (`"CO (ppm)"`), los demás sin unidades (`"CO"`).
Si un año tiene 0 en la primera columna, el imputador lo vio vacío.

In [ ]:
con_unidades = [c for c in ni.columns if " (" in c]
pd.DataFrame({
    "datos en columnas sin unidades": ni.groupby("Archivo_Origen")[VARS].count().sum(axis=1),
    "datos en columnas con unidades": ni.groupby("Archivo_Origen")[con_unidades].count().sum(axis=1),
    "filas": ni.groupby("Archivo_Origen").size(),
})

## 3. Juntar las columnas y alinear las dos bases
Pasamos los valores de `"CO (ppm)"` a `"CO"`, etc. Así recuperamos los datos reales de 2024.
Luego unimos por estación y fecha: cada variable queda como `_real` y `_imp`.

In [ ]:
for c in con_unidades:
    base = c.split(" (")[0]
    ni[base] = ni[base].fillna(ni[c])

for df in (ni, im):
    df["Fecha y hora"] = pd.to_datetime(df["Fecha y hora"])

m = ni[["Estacion", "Fecha y hora"] + VARS].merge(
        im, on=["Estacion", "Fecha y hora"], suffixes=("_real", "_imp"))
m["año"] = m["Fecha y hora"].dt.year
print(m.shape)

## 4. ¿Se conservaron los valores observados?
Donde había dato real, la base imputada debería traer exactamente el mismo valor (0 % cambiados).

In [ ]:
def pct_cambiados(g, v):
    k = g[v + "_real"].notna()
    return 100 * ((g.loc[k, v + "_real"] - g.loc[k, v + "_imp"]).abs() > 1e-3).mean()

m.groupby("año").apply(lambda g: pd.Series({v: pct_cambiados(g, v) for v in ["O3", "PM10", "SO2"]})).round(1)

## 5. 2024 como prueba: real contra imputado
El imputador creyó que 2024 estaba vacío, pero tenemos los valores reales.
Es una validación natural: si la imputación fuera buena, la correlación sería alta y los extremos se parecerían.

In [ ]:
a = m[m["año"] == 2024]
filas = {}
for v in VARS:
    k = a[v + "_real"].notna()
    r, i = a.loc[k, v + "_real"], a.loc[k, v + "_imp"]
    filas[v] = {"correlación": r.corr(i), "media real": r.mean(), "media imputada": i.mean(),
                "sesgo": (i - r).mean(), "P99 real": r.quantile(0.99), "P99 imputado": i.quantile(0.99)}
pd.DataFrame(filas).T.round(2)

## 6. ¿Cuántos excesos ve el modelo?
La red solo aprende de los excesos sobre el umbral (O3: 38 ppb, SO2: 20.2 ppb).

In [ ]:
UMBRALES = {"O3": 38.0, "SO2": 20.2}
pd.concat({v: m.groupby("año").apply(lambda g: pd.Series({
              "real": (g[v + "_real"] > u).sum(),
              "base imputada": (g[v + "_imp"] > u).sum()}))
           for v, u in UMBRALES.items()}, axis=1)

## 7. Valores físicamente imposibles en la base imputada

In [ ]:
I = {v: m[v + "_imp"] for v in VARS}
pd.Series({
    "Lluvia negativa": (I["RAINF"] < 0).sum(),
    "Radiación negativa": (I["SR"] < 0).sum(),
    "Contaminantes negativos": sum((I[v] < 0).sum() for v in ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "SO2"]),
    "RH > 100 %": (I["RH"] > 100).sum(),
    "PM2.5 > PM10": (I["PM2.5"] > I["PM10"]).sum(),
    "Temperatura fuera de [-10, 50]": ((I["TOUT"] < -10) | (I["TOUT"] > 50)).sum(),
    "PM10 >= 999": (I["PM10"] >= 999).sum(),
}, name="casos")

## 8. ¿La imputación aplana los extremos? (fuera de 2024)
Comparamos los valores observados contra los que el modelo rellenó en los huecos.

In [ ]:
fuera = m["año"] != 2024
filas = {}
for v in ["O3", "PM10", "PM2.5", "NO2", "SO2", "CO"]:
    obs = m.loc[fuera, v + "_real"].dropna()
    imp = m.loc[fuera & m[v + "_real"].isna(), v + "_imp"]
    filas[v] = {"n imputados": len(imp), "media obs": obs.mean(), "media imp": imp.mean(),
                "P99 obs": obs.quantile(0.99), "P99 imp": imp.quantile(0.99), "máx imp": imp.max()}
pd.DataFrame(filas).T.round(2)